In [1]:
import pandas as pd
import json

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Load the Excel file
beverage_list_path = './TCF-Coffe_App-Export_Beverage-List.csv'
impact_data_path = './TCF-Coffe_App-Export_Impact-Data.csv'
sugar_data_path = './TCF-Coffe_App-Export_Sugar-Data.csv'
impact_description_path = './TCF-Coffe_App-Export_Impact-Description.csv'
coffee_description_path = './TCF-Coffe_App-Export_Coffee-Description.csv'

df_beverage_list = pd.read_csv(beverage_list_path,delimiter=';',encoding='windows-1252')
df_impact_data = pd.read_csv(impact_data_path,delimiter=';',encoding='windows-1252')
df_sugar_data = pd.read_csv(sugar_data_path,delimiter=';',encoding='windows-1252')
df_impact_description = pd.read_csv(impact_description_path,delimiter=';',encoding='windows-1252')
df_coffee_description = pd.read_csv(coffee_description_path,delimiter=';',encoding='windows-1252')

# df_beverage_list['Labels'] = df_beverage_list['Labels'].str.split(';').str[0]
# df_beverage_list['Labels'] = df_beverage_list['Labels'].str.replace(' ', '', regex=False).str.replace(',', '|', regex=False)
df_beverage_list['Retail name'] = df_beverage_list['Retail name'] + " " + df_beverage_list['Salepoint']

df_beverage_list

,Beverage ID,Retail name,Retail price,Hidden Costs,True Price,Beverage-type,Salepoint,Deca,Milk Type,Price Excluding Tax,Value Added Tax,Smart Value Added Tax,Smart Pricing rounded,Labels
0,"Café, décaféiné, Le Klee",Café (deca) Le Klee,1.6,0.292,1.89,Café,Le Klee,True,none,1.480,0.120,0.057,1.55,none
1,"Cappuccino, décaféiné, Le Klee",Cappuccino (deca) Le Klee,2.7,0.436,3.14,Cappuccino,Le Klee,True,Cow milk,2.498,0.202,0.072,2.70,none
2,"Cappuccino, décaféiné, lait d'amande, Le Klee",Cappuccino (deca) Le Klee,2.7,0.444,3.14,Cappuccino,Le Klee,True,Almond milk,2.498,0.202,0.073,2.70,none
3,"Cappuccino, décaféiné, lait d'avoine, Le Klee",Cappuccino (deca) Le Klee,2.7,0.338,3.04,Cappuccino,Le Klee,True,Oat milk,2.498,0.202,0.062,2.65,eu-organic
4,"Cappuccino, décaféiné, lait de soja, Le Klee",Cappuccino (deca) Le Klee,2.7,0.474,3.17,Cappuccino,Le Klee,True,Soya milk,2.498,0.202,0.076,2.70,none
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,"Renversé, lait d'avoine, Le Klee",Renversé Via Verde Le Klee,2.7,0.123,2.82,Renversé,Le Klee,False,Oat milk,2.498,0.202,0.039,2.60,via-verde|fairtrade|eu-organic|eu-organic
59,"Renversé, lait de soja, Le Klee",Renversé Via Verde Le Klee,2.7,0.293,2.99,Renversé,Le Klee,False,Soya milk,2.498,0.202,0.057,2.65,via-verde|fairtrade|eu-organic
60,"Renversé, sans-lactose, Le Klee",Renversé Via Verde Le Klee,2.7,0.245,2.95,Renversé,Le Klee,False,Cow milk,2.498,0.202,0.052,2.65,via-verde|fairtrade|eu-organic
61,"Ristretto, Compass Machine",Ristretto Via Verde Compass Machine,1.5,0.065,1.57,Ristretto,Compass Machine,False,none,1.388,0.112,0.033,1.45,via-verde|fairtrade|eu-organic


In [4]:

duplicate_rows = df_impact_data.groupby(by=["Beverage ID", "Ingredient", "Impact Category", "Stage", "Indicator"]).filter(lambda x: len(x) >= 2)
duplicate_rows = duplicate_rows.sort_values(by=["Beverage ID", "Ingredient", "Impact Category", "Stage", "Indicator"])
duplicate_rows.to_csv('duplicate_rows.csv',index=False)

In [5]:
duplicate_rows

,Beverage ID,Ingredient,Stage,Impact Category,Indicator,Unit,Monetary Value,Value,References
10457,"Café Macchiato Blue Planet, Dallmayr",Cow milk,Transport,Environment,Fine particulate matter formation,kg PM10 eq,0.0000,8.715723e-07,"Road; transport, freight, lorry 3.5-7.5 metric..."
10458,"Café Macchiato Blue Planet, Dallmayr",Cow milk,Transport,Environment,Fine particulate matter formation,kg PM2.5 eq,0.0001,9.245213e-07,"Road; transport, freight, lorry with refrigera..."
10483,"Café Macchiato Via Verde, Dallmayr",Cow milk,Transport,Environment,Fine particulate matter formation,kg PM10 eq,0.0000,8.715723e-07,"Road; transport, freight, lorry 3.5-7.5 metric..."
10484,"Café Macchiato Via Verde, Dallmayr",Cow milk,Transport,Environment,Fine particulate matter formation,kg PM2.5 eq,0.0001,9.245213e-07,"Road; transport, freight, lorry with refrigera..."
10509,"Cappuccino Blue Planet, Dallmayr",Cow milk,Transport,Environment,Fine particulate matter formation,kg PM10 eq,0.0000,1.030040e-06,"Road; transport, freight, lorry 3.5-7.5 metric..."
...,...,...,...,...,...,...,...,...,...
11474,"Renversé, décaféiné, Le Klee",Cow milk,Transport,Environment,Fine particulate matter formation,kg PM2.5 eq,0.0019,1.764995e-05,"Road; transport, freight, lorry with refrigera..."
11553,"Renversé, décaféiné, sans-lactose, Le Klee",Cow milk,Transport,Environment,Fine particulate matter formation,kg PM10 eq,0.0000,3.634976e-05,"Road; transport, freight, light commercial veh..."
11554,"Renversé, décaféiné, sans-lactose, Le Klee",Cow milk,Transport,Environment,Fine particulate matter formation,kg PM2.5 eq,0.0019,1.764995e-05,"Road; transport, freight, lorry with refrigera..."
11633,"Renversé, sans-lactose, Le Klee",Cow milk,Transport,Environment,Fine particulate matter formation,kg PM10 eq,0.0000,3.634976e-05,"Road; transport, freight, light commercial veh..."


In [6]:
def get_indicator_values(row):

    # # Find matches where df_impact_description['Indicator'] is contained in row['Indicator']
    # matched_rows = df_impact_description[
    #     df_impact_description['Indicator'].apply(lambda x: x in row['Indicator'])
    # ]
    
    # if matched_rows.empty:
    #     impact_definition = None
    #     monetisation_method = None
    # else:
    #     # Select the most specific match (longest string in df_impact_description['Indicator'])
    #     impact_definition = matched_rows.loc[matched_rows['Indicator'].str.len().idxmax(), 'Indicator definition']
    #     monetisation_method = matched_rows.loc[matched_rows['Indicator'].str.len().idxmax(), 'Monetisation method']
    
    return {
        'indicators': row['Indicator'],
        'unit': row['Unit'],
        'impactValue': row['Value'], 
        'costValue': row['Monetary Value'], 
        # 'impactDefinition': impact_definition,
        # 'monetisationMethod': monetisation_method,
        'reference': None if pd.isna(row['References']) else row['References']
    }


def calculate_impacts(group):
    # Group by stage and impact-category
    grouped_impacts = group.groupby(['Ingredient','Stage', 'Impact Category']).apply(lambda x: {
        'stage': x.iloc[0]['Stage'],
        'ingredient': x.iloc[0]['Ingredient'],
        'impactCategory': x.iloc[0]['Impact Category'],
        'impactValue': x['Value'].sum(),   # Sum impact values
        'costValue': x['Monetary Value'].sum(),   # Sum cost values
        'details': x.apply(get_indicator_values, axis=1).tolist()
    }).reset_index(drop=True).tolist()

    return grouped_impacts

In [7]:
df_beverage_list.head()

,Beverage ID,Retail name,Retail price,Hidden Costs,True Price,Beverage-type,Salepoint,Deca,Milk Type,Price Excluding Tax,Value Added Tax,Smart Value Added Tax,Smart Pricing rounded,Labels
0,"Café, décaféiné, Le Klee",Café (deca) Le Klee,1.6,0.292,1.89,Café,Le Klee,True,none,1.480,0.120,0.057,1.55,none
1,"Cappuccino, décaféiné, Le Klee",Cappuccino (deca) Le Klee,2.7,0.436,3.14,Cappuccino,Le Klee,True,Cow milk,2.498,0.202,0.072,2.70,none
2,"Cappuccino, décaféiné, lait d'amande, Le Klee",Cappuccino (deca) Le Klee,2.7,0.444,3.14,Cappuccino,Le Klee,True,Almond milk,2.498,0.202,0.073,2.70,none
3,"Cappuccino, décaféiné, lait d'avoine, Le Klee",Cappuccino (deca) Le Klee,2.7,0.338,3.04,Cappuccino,Le Klee,True,Oat milk,2.498,0.202,0.062,2.65,eu-organic
4,"Cappuccino, décaféiné, lait de soja, Le Klee",Cappuccino (deca) Le Klee,2.7,0.474,3.17,Cappuccino,Le Klee,True,Soya milk,2.498,0.202,0.076,2.70,none


In [8]:
df_beverage_list['Milk Type'].unique()


array(['none', 'Cow milk', 'Almond milk', 'Oat milk', 'Soya milk'],
      dtype=object)

In [9]:
# Transform the data
import os
import re


formatted_data = []

i = 0;

# milk_beverage_lists = ['Cappuccino',"Chocolait","Latte Macchiato","Macchiato","Mocaccino", 'Renversé']

for _, beverage in df_beverage_list.iterrows():
    beverage_id = beverage['Beverage ID']
    retail_name = beverage['Retail name']

    # all_removals = labels_list + ['décaféiné', "lait d'amande", "lait d'avoine", "lait de soja", "sans-lactose"]

    # recipe_id_pattern = r'(' + '|'.join(map(re.escape, all_removals)) + r')'
    # recipe_id = re.sub(recipe_id_pattern, '', retail_name, flags=re.IGNORECASE).replace(',','').strip()
    recipe_id = beverage['Beverage-type']
    # sale_point_id = sale_point_mapping[beverage['Salepoint']]
    # is_decaf = 'décaféiné' in beverage['Retail name'].lower()
    is_decaf = beverage['Deca']
    # has_milk = any(x in beverage['Retail name'] for x in milk_beverage_lists)

    # Determine the type of milk, if present
    # milk_types = {'lait d\'amande': 'Almond', 'lait d\'avoine': 'Oat', 'lait de soja': 'Soy', 'sans-lactose': 'Lactose-Free'}
    # milk_type = None
    # for key, value in milk_types.items():
    #     if key in beverage['Retail name'].lower():
    #         milk_type = value
    #         break
    # milk_type = milk_type if milk_type else 'Dairy' if any(x in beverage['Retail name'].lower() for x in ['milk', 'latte', 'cappuccino', 'renversé']) else None
        
    milk_type = beverage['Milk Type']
    has_milk = milk_type != 'none'

    definition_row = df_coffee_description[df_coffee_description['Recipe'] == recipe_id]
    definition = definition_row['Definition'].iloc[0] if not definition_row.empty else ""


    impact_rows = df_impact_data[df_impact_data['Beverage ID'] == beverage_id]
    # print(impact_rows)
    ingredient_list = impact_rows['Ingredient'].unique().tolist()



    grouped_impacts = calculate_impacts(impact_rows)

    path_impacts = './results/impacts/'+beverage_id.lower().replace(' ','_').replace(',','')+'.json'
    os.makedirs(os.path.dirname(path_impacts), exist_ok=True)
    with open(path_impacts, 'w', encoding='utf-8') as f:
        json.dump(grouped_impacts, f, indent=4, ensure_ascii=False)



    formatted_data.append({
        'serveId': beverage_id,
        'recipeId': recipe_id,
        'retailName': retail_name,
        'retailPrice': beverage['Retail price'],
        'hiddenCost': beverage['Hidden Costs'],
        'truePrice': beverage['True Price'],
        'labels': beverage['Labels'],
        "priceWithoutTax": beverage["Price Excluding Tax"],
        'valueAddedTax': beverage["Value Added Tax"],
        'smartValueAddedTax': beverage["Smart Value Added Tax"],
        'smartPricingRounded': beverage["Smart Pricing rounded"],
        'isDecaf': is_decaf,
        'hasMilk': has_milk,
        'milkType': milk_type,
        'coffeeDetails': definition, 

    })

# Create the formatted DataFrame
formatted_df = pd.DataFrame(formatted_data)

# Save to CSV
output_path = './results/coffee_data.csv'
formatted_df.to_csv(output_path, index=False)



print(f"Formatted data saved to {output_path}")

Formatted data saved to ./results/coffee_data.csv


In [10]:

list_sugars = ["Swiss sugar default","Swiss sugar low","Swiss sugar moderate","Swiss sugar high"]

for sugar in list_sugars:
    sugar_rows = df_sugar_data[df_sugar_data['Beverage ID'] == sugar]
    print(sugar_rows)

    grouped_impacts = calculate_impacts(sugar_rows)

    path_impacts = './results/sugar/'+sugar.lower().replace(' ','_').replace(',','')+'.json'
    os.makedirs(os.path.dirname(path_impacts), exist_ok=True)
    with open(path_impacts, 'w', encoding='utf-8') as f:
        json.dump(grouped_impacts, f, indent=4, ensure_ascii=False)


             Beverage ID      Country Ingredient Labels        Stage  \
0    Swiss sugar default  Switzerland  Sugarbeet   none   Production   
1    Swiss sugar default  Switzerland  Sugarbeet   none   Production   
2    Swiss sugar default  Switzerland  Sugarbeet   none   Production   
3    Swiss sugar default  Switzerland  Sugarbeet   none   Production   
4    Swiss sugar default  Switzerland  Sugarbeet   none   Production   
..                   ...          ...        ...    ...          ...   
189  Swiss sugar default  Switzerland  Sugarbeet   none  Consumption   
190  Swiss sugar default  Switzerland  Sugarbeet   none  Consumption   
191  Swiss sugar default  Switzerland  Sugarbeet   none  Consumption   
192  Swiss sugar default  Switzerland  Sugarbeet   none  Consumption   
193  Swiss sugar default  Switzerland  Sugarbeet   none  Consumption   

    Impact Category                          Indicator         Unit  Value  \
0       Environment  Fine particulate matter formation  k

In [11]:
output_path= "./results/impacts_definitions.csv"
df_impact_description.to_csv(output_path, index=False)

In [12]:
import shutil

source_dir = './results/'
destination_dir = '../public/data/'

# Copy the contents of the source directory to the destination directory
shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)

print(f"Contents of {source_dir} copied to {destination_dir}")

Contents of ./results/ copied to ../public/data/
